# Kafka batching analysis plots

This notebook draws plots from the regenerative analysis CSV files:

- `steady-results.csv`
- `transient-results.csv`

Run the Java analyses from the project root before running this notebook.

## 1. Create the CSV files

```bash
mkdir -p outcomes

mvn -q org.codehaus.mojo:exec-maven-plugin:3.5.0:java \
  -Dexec.mainClass=com.myexam.qesm.analysis.RegenerativeSteadyState \
  > outcomes/steady-results.csv

mvn -q org.codehaus.mojo:exec-maven-plugin:3.5.0:java \
  -Dexec.mainClass=com.myexam.qesm.analysis.RegenerativeTransient \
  > outcomes/transient-results.csv
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.title_fontsize": 10,
    "legend.fontsize": 9,
})

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pom.xml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project folder containing pom.xml")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTCOMES_DIR = PROJECT_ROOT / "outcomes"
STEADY_CSV = OUTCOMES_DIR / "steady-results.csv"
TRANSIENT_CSV = OUTCOMES_DIR / "transient-results.csv"
PLOTS_DIR = OUTCOMES_DIR / "plots"
SAVE_PLOTS = False

def finish_plot(fig, filename):
    fig.tight_layout()
    if SAVE_PLOTS:
        PLOTS_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(PLOTS_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()

## 2. Load and check the results

In [ ]:
def load_results(path, required_columns):
    if not path.exists():
        raise FileNotFoundError(
            f"{path.name} was not found. Run the related Maven command first."
        )

    data = pd.read_csv(path)
    if data.empty:
        raise ValueError(f"{path.name} contains no result rows")
    missing = sorted(set(required_columns) - set(data.columns))
    if missing:
        raise ValueError(f"{path.name} is missing columns: {missing}")

    source_files = list((PROJECT_ROOT / "src/main/java").rglob("*.java"))
    if source_files and path.stat().st_mtime < max(file.stat().st_mtime for file in source_files):
        print(f"Warning: {path.name} is older than the Java model. Regenerate it before final plots.")
    return data

steady_required = {
    "lambda", "timeout", "B", "probability_sum",
    "effective_arrival_rate", "average_buffer",
    "batch_throughput", "message_throughput",
    "average_batch_size", "queue_waiting_time",
    "total_system_time",
}
transient_required = {
    "lambda", "timeout", "B", "time", "probability_sum",
    "p_broker_busy", "p_timer_active", "average_buffer",
    "average_in_service", "effective_arrival_rate",
    "batch_completion_rate", "message_completion_rate",
    "cumulative_batches", "cumulative_messages",
}

steady = load_results(STEADY_CSV, steady_required)
transient = load_results(TRANSIENT_CSV, transient_required)

print(f"Steady-state rows: {len(steady)}")
print(f"Transient rows: {len(transient)}")
display(steady.head())
display(transient.head())

In [ ]:
steady_probability_error = (steady["probability_sum"] - 1.0).abs().max()
transient_probability_error = (transient["probability_sum"] - 1.0).abs().max()
throughput_difference = (
    steady["effective_arrival_rate"] - steady["message_throughput"]
).abs().max()

print(f"Maximum steady probability error: {steady_probability_error:.3e}")
print(f"Maximum transient probability error: {transient_probability_error:.3e}")
print(f"Maximum steady flow difference: {throughput_difference:.3e}")

## 3. Steady-state plots

Each line represents one timeout value. The horizontal axis shows the arrival rate $\lambda$.

In [ ]:
steady_metrics = [
    ("average_batch_size", "Average batch size", "Messages per batch"),
    ("queue_waiting_time", "Average queue waiting time", "Time units"),
    ("message_throughput", "Message throughput", "Messages per time unit"),
    ("average_buffer", "Average buffer occupancy", "Messages"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (column, title, ylabel) in zip(axes.flat, steady_metrics):
    for timeout, group in steady.sort_values("lambda").groupby("timeout"):
        ax.plot(
            group["lambda"], group[column], marker="o",
            linewidth=2, label=f"T = {timeout:g}"
        )
    ax.set_title(title)
    ax.set_xlabel("Arrival rate λ")
    ax.set_ylabel(ylabel)
    ax.legend(title="Timeout")

fig.suptitle("Regenerative steady-state results", fontsize=16, y=1.02)
finish_plot(fig, "steady_state_metrics.png")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for timeout, group in steady.sort_values("lambda").groupby("timeout"):
    ax.plot(
        group["lambda"], group["batch_throughput"],
        marker="o", linewidth=2, label=f"Batch throughput, T={timeout:g}"
    )
    ax.plot(
        group["lambda"], group["message_throughput"],
        marker="s", linestyle="--", linewidth=2,
        label=f"Message throughput, T={timeout:g}"
    )

ax.set_title("Batch throughput and message throughput")
ax.set_xlabel("Arrival rate λ")
ax.set_ylabel("Completions per time unit")
ax.legend(ncol=2)
finish_plot(fig, "steady_state_throughput.png")

## 4. Transient plots

Each line represents one arrival rate. These plots show how the model changes after starting from an empty buffer.

In [ ]:
transient_metrics = [
    ("average_buffer", "Average buffer occupancy", "Messages"),
    ("p_broker_busy", "Probability that the broker is busy", "Probability"),
    ("message_completion_rate", "Message completion rate", "Messages per time unit"),
    ("cumulative_messages", "Cumulative completed messages", "Messages"),
]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (column, title, ylabel) in zip(axes.flat, transient_metrics):
    for arrival_rate, group in transient.sort_values("time").groupby("lambda"):
        ax.plot(
            group["time"], group[column], linewidth=2,
            label=f"λ = {arrival_rate:g}"
        )
    ax.set_title(title)
    ax.set_xlabel("Time")
    ax.set_ylabel(ylabel)
    ax.legend(title="Arrival rate")

fig.suptitle("Regenerative transient results", fontsize=16, y=1.02)
finish_plot(fig, "transient_metrics.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for arrival_rate, group in transient.sort_values("time").groupby("lambda"):
    axes[0].plot(
        group["time"], group["p_timer_active"],
        linewidth=2, label=f"λ = {arrival_rate:g}"
    )
    axes[1].plot(
        group["time"], group["average_in_service"],
        linewidth=2, label=f"λ = {arrival_rate:g}"
    )

axes[0].set_title("Probability that the timer is active")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Probability")
axes[1].set_title("Average messages in service")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Messages")
for ax in axes:
    ax.legend(title="Arrival rate")

finish_plot(fig, "transient_timer_and_service.png")

## 5. Buffer-size probability plot

Choose one arrival rate below. For the current model, the expected columns are `p_buffer_0` to `p_buffer_3`. The notebook also checks whether any probability is missing from these four buffer states.

In [ ]:
available_arrival_rates = sorted(transient["lambda"].unique())
SELECTED_LAMBDA = available_arrival_rates[len(available_arrival_rates) // 2]

buffer_columns = sorted(
    [column for column in transient.columns if column.startswith("p_buffer_")],
    key=lambda column: int(column.rsplit("_", 1)[1]),
)
selected = transient[transient["lambda"] == SELECTED_LAMBDA].sort_values("time").copy()
selected["other_buffer_sizes"] = (
    selected["probability_sum"] - selected[buffer_columns].sum(axis=1)
).clip(lower=0.0)

plot_columns = buffer_columns.copy()
labels = [column.replace("p_buffer_", "Buffer = ") for column in buffer_columns]
if selected["other_buffer_sizes"].max() > 1e-9:
    plot_columns.append("other_buffer_sizes")
    labels.append("Other buffer sizes")
    print(
        "Warning: the CSV does not report every buffer-size probability. "
        f"Maximum unreported probability: {selected['other_buffer_sizes'].max():.4f}"
    )

fig, ax = plt.subplots(figsize=(11, 6))
ax.stackplot(
    selected["time"],
    *[selected[column] for column in plot_columns],
    labels=labels, alpha=0.85,
)
ax.set_title(f"Buffer-size probabilities, λ = {SELECTED_LAMBDA:g}")
ax.set_xlabel("Time")
ax.set_ylabel("Probability")
ax.set_ylim(0, 1)
ax.legend(loc="upper right", ncol=2)
finish_plot(fig, "transient_buffer_probabilities.png")

## 6. Save the figures

Set `SAVE_PLOTS = True` in the first code cell and run the plotting cells again. PNG files will be saved in the `outcomes/plots/` folder.